# CITE-seq RNA deconvolution demo

Train DECIPHER on a shared PBMC RNA reference and three held-out donor synthetic bulks (HS12 / HS13 / HS15).

**Data** (under `data/citeseq/`):
- `sc_ref_counts.csv`, `sc_ref_meta.csv` — single-cell reference (genes × cells)
- `HS12|HS13|HS15/bulk_counts.csv`, `bulk_meta.csv` — real mixtures with ground-truth proportions



In [1]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from torch.utils.data import DataLoader

REPO_ROOT = Path(".").resolve()
from DECIPHER import (
    DECIPHER,
    DecodePseudoBulkBuilder,
    DecodePseudoBulkConfig,
    LossWeights,
    PseudoRealDynamicPairDataset,
    ccc,
    pair_collate,
    process_adata,
    reorder_celltype_proportions,
    row_max_normalize,
    seed_everything,
)

DATA = REPO_ROOT / "data" / "citeseq"
OUTDIR = Path(".").resolve() / "runs_citeseq"
OUTDIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUTDIR / "DECIPHER_citeseq.pt"

CELLTYPE_ORDER = ["CD4T", "Myeloid", "CD8T", "B cells", "NK"]
BATCHES = ["HS12", "HS15", "HS13"]
N_PSEUDO = 8000
CELLS_PER_BULK = 200
PSEUDO_VAL_SPLIT = 0.2
BATCH_SIZE = 128
MAX_EPOCH = 300
PATIENCE = 10
SEED_TRAIN, SEED_VAL = 41, 43
N_DOMAIN = 2

seed_everything(42)
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| DATA:", DATA)

LOSS_W = LossWeights(
    w_prop=100, w_rec=0.1, w_latrec=0.1, w_dom=0.2,
    w_align=0.3, w_contrast=0.1, mse_ce_ratio=20,
)


device: cuda | DATA: /data1/lcy/decipher/DECIPHER_github/data/citeseq


In [2]:
# Load single-cell reference (genes × cells) → AnnData (cells × genes)
sc_counts = pd.read_csv(DATA / "sc_ref_counts.csv", index_col=0)  # genes × cells
sc_meta = pd.read_csv(DATA / "sc_ref_meta.csv", index_col=0)
common_cells = [c for c in sc_counts.columns.astype(str) if c in sc_meta.index.astype(str)]
sc_counts = sc_counts.loc[:, common_cells]
sc_meta = sc_meta.loc[common_cells]
sc_meta["celltype"] = sc_meta["celltype"].astype(str)
keep = sc_meta["celltype"].isin(CELLTYPE_ORDER).values
sc_counts = sc_counts.loc[:, keep]
sc_meta = sc_meta.loc[keep]

adata = ad.AnnData(
    X=sc_counts.T.values.astype(np.float32),
    obs=sc_meta.copy(),
    var=pd.DataFrame(index=sc_counts.index.astype(str)),
)
print("sc ref:", adata.shape, adata.obs["celltype"].value_counts().to_dict())

sc ref: (11012, 1970) {'CD4T': 4161, 'Myeloid': 2367, 'CD8T': 1986, 'NK': 1304, 'B cells': 1194}


In [3]:
# Load real bulks (genes × samples) + ground-truth proportions
gene_names = list(adata.var_names.astype(str))
bulk_blocks, prop_blocks = [], []

for donor in BATCHES:
    bdir = DATA / donor
    bulk = pd.read_csv(bdir / "bulk_counts.csv", index_col=0)  # genes × samples
    meta = pd.read_csv(bdir / "bulk_meta.csv", index_col=0)
    shared = [g for g in gene_names if g in bulk.index.astype(str)]
    X = bulk.loc[shared, :].T.values.astype(np.float32)  # samples × genes
    sample_ids = bulk.columns.astype(str)
    props = np.stack([
        np.array([float(meta.loc[sid, ct]) for ct in CELLTYPE_ORDER], dtype=np.float32)
        for sid in sample_ids
    ])
    props = props / props.sum(axis=1, keepdims=True)
    bulk_blocks.append(X)
    prop_blocks.append(props)
    print(donor, "bulk", X.shape)

gene_names = shared
adata = adata[:, gene_names].copy()
bulk_raw_all = np.vstack(bulk_blocks).astype(np.float32)
prop_real_all = np.vstack(prop_blocks).astype(np.float32)
real_domain = np.ones(bulk_raw_all.shape[0], dtype=np.int64)
G = len(gene_names)
print("bulk_raw_all", bulk_raw_all.shape, "G", G)

HS12 bulk (1000, 1970)
HS15 bulk (1000, 1970)
HS13 bulk (1000, 1970)
bulk_raw_all (3000, 1970) G 1970


In [4]:
# Prototypes (PCA) + pseudo mixtures from the sc reference
sc_proc = process_adata(adata, celltype_key="celltype", state_key=None, assume_log1p_cp10k=False)
X_sc = np.asarray(sc_proc.X, dtype=np.float32)
Z = PCA(n_components=min(16, X_sc.shape[0], X_sc.shape[1]), random_state=42).fit_transform(X_sc)
labels = adata.obs["celltype"].astype(str).to_numpy()
s_z = torch.from_numpy(
    np.stack([Z[labels == ct].mean(0) for ct in CELLTYPE_ORDER]).astype(np.float32)
)

builder = DecodePseudoBulkBuilder(
    sc_proc, DecodePseudoBulkConfig(cells_per_bulk=CELLS_PER_BULK, seed=0), require_state=False,
)
Xs, ps = [], []
for _ in range(N_PSEUDO):
    x, p, _ = builder.build_one()
    Xs.append(x)
    ps.append(reorder_celltype_proportions(p, sc_proc.celltype_map, CELLTYPE_ORDER))
X_pseudo_raw = np.stack(Xs)
props_pseudo = np.stack(ps).astype(np.float32)
print("pseudo", X_pseudo_raw.shape, "s_z", tuple(s_z.shape))

idx_train, idx_val = train_test_split(
    np.arange(N_PSEUDO), test_size=PSEUDO_VAL_SPLIT, random_state=42,
)

pseudo (8000, 1970) s_z (5, 16)


In [5]:
# Normalize, train, deconvolve held-out real bulks
X_pseudo = row_max_normalize(X_pseudo_raw)
bulk_X = row_max_normalize(bulk_raw_all)

train_ds = PseudoRealDynamicPairDataset(
    X_pseudo[idx_train], props_pseudo[idx_train], bulk_X, real_domain, seed=SEED_TRAIN,
)
val_ds = PseudoRealDynamicPairDataset(
    X_pseudo[idx_val], props_pseudo[idx_val], bulk_X, real_domain,
    seed=SEED_VAL, real_sampling="fixed_cycle",
)
g = torch.Generator()
g.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pair_collate, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pair_collate)

seed_everything(42)
model = DECIPHER(s_z=s_z, n_feature=G, hidden_dim=(128, 128), n_domain=N_DOMAIN).to(device)
model.fit(
    train_loader, val_dataloader=val_loader, lr=1e-4, weight_decay=1e-3,
    max_epoch=MAX_EPOCH, device=device, patience=PATIENCE, loss_weight=LOSS_W,
    outdir=str(CKPT_PATH), verbose=True,
)

_, proportions = model.deconvolution(torch.from_numpy(bulk_X).to(device))
prop_pred = proportions.cpu().numpy()

offset = 0
rows = []
for donor, blk in zip(BATCHES, bulk_blocks):
    n = blk.shape[0]
    yt, yp = prop_real_all[offset:offset + n], prop_pred[offset:offset + n]
    offset += n
    for j, ct in enumerate(CELLTYPE_ORDER):
        r, _ = pearsonr(yt[:, j], yp[:, j])
        rows.append({
            "donor": donor, "celltype": ct,
            "ccc": ccc(yt[:, j], yp[:, j]),
            "rmse": float(np.sqrt(((yt[:, j] - yp[:, j]) ** 2).mean())),
            "pearson": float(r),
        })
    print(donor, "overall CCC", ccc(yt, yp))

metrics = pd.DataFrame(rows)
metrics.to_csv(OUTDIR / "metrics_by_celltype.csv", index=False)
pd.DataFrame(prop_pred, columns=CELLTYPE_ORDER).to_csv(OUTDIR / "pred_proportions.csv", index=False)
display(metrics.groupby("donor")[["ccc", "rmse", "pearson"]].mean())
print("saved under", OUTDIR)

/data1/lcy/decipher/DECIPHER_github/DECIPHER/model.py:354: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#cublasApi_reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:156.)
  z_hat = pi @ S                      # [B, dc]
/data1/lcy/decipher/DECIPHER_github/DECIPHER/model.py:355: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >

[Epoch 001/300] train: total=2.4882 prop=0.0186 rec=0.0541 latrec=0.0071 domain=0.6833 align=0.0732 contrast=4.8414 | val: total=1.4225 prop=0.0081 rec=0.0204 latrec=0.0073 domain=0.6528 align=0.4007 contrast=4.7485
New best total loss. Model saved.
[Epoch 002/300] train: total=1.4383 prop=0.0084 rec=0.0165 latrec=0.0076 domain=0.6242 align=0.0811 contrast=4.7507 | val: total=1.0197 prop=0.0045 rec=0.0070 latrec=0.0080 domain=0.5210 align=0.2802 contrast=4.6177
New best total loss. Model saved.
[Epoch 003/300] train: total=1.0829 prop=0.0053 rec=0.0091 latrec=0.0085 domain=0.4199 align=0.1424 contrast=4.6400 | val: total=0.7631 prop=0.0028 rec=0.0049 latrec=0.0089 domain=0.1855 align=0.3249 contrast=4.4823
New best total loss. Model saved.
[Epoch 004/300] train: total=0.9161 prop=0.0037 rec=0.0064 latrec=0.0096 domain=0.2920 align=0.0886 contrast=4.5643 | val: total=0.6728 prop=0.0016 rec=0.0038 latrec=0.0106 domain=0.2054 align=0.1022 contrast=4.3982
New best total loss. Model saved.


,ccc,rmse,pearson
donor,,,
HS12,0.943184,0.035961,0.961623
HS13,0.954157,0.034528,0.960150
HS15,0.935418,0.040659,0.959110


saved under /data1/lcy/decipher/DECIPHER_github/runs_citeseq
